# ABCD Dataset Download + FinDisputeEval EDA

This notebook is intended for VS Code with a Colab runtime. It mounts Google Drive, downloads the official `asappresearch/abcd` files, normalizes ABCD conversations, and runs EDA focused on FinDisputeEval use cases.

ABCD is not a banking-dispute corpus. Its value for FinDisputeEval is policy-constrained multi-turn customer support: action sequencing, slot collection, escalation, denial/explanation, and handoff structure.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%pip -q install -U pandas pyarrow matplotlib

In [ ]:
from __future__ import annotations

import gzip
import json
import math
import os
import re
import urllib.request
from datetime import datetime, timezone
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "FinDisputeEval"
RUN_ID = globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RAW_DIR = PROJECT_DIR / "dataset" / "external" / "abcd" / "raw"
PROCESSED_DIR = PROJECT_DIR / "dataset" / "interim" / "abcd" / "processed_findispute"
EDA_DIR = PROJECT_DIR / "outputs" / "data_pipeline" / "abcd_structure_eda" / "eda_v01" / f"run_{RUN_ID}_colab"
OUTPUT_DIR = EDA_DIR

for directory in [RAW_DIR, PROCESSED_DIR, EDA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

REPO_BASE = "https://raw.githubusercontent.com/asappresearch/abcd/master/data"
ABCD_FILES = {
    "abcd_v1.1.json.gz": f"{REPO_BASE}/abcd_v1.1.json.gz",
    "abcd_sample.json": f"{REPO_BASE}/abcd_sample.json",
    "guidelines.json": f"{REPO_BASE}/guidelines.json",
    "kb.json": f"{REPO_BASE}/kb.json",
    "ontology.json": f"{REPO_BASE}/ontology.json",
    "utterances.json": f"{REPO_BASE}/utterances.json",
}

RANDOM_STATE = 42

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
def download_if_missing(url: str, target_path: Path) -> None:
    if target_path.exists() and target_path.stat().st_size > 0:
        print(f"Already exists: {target_path.name} ({target_path.stat().st_size:,} bytes)")
        return
    print(f"Downloading {target_path.name} ...")
    urllib.request.urlretrieve(url, target_path)
    print(f"Saved: {target_path} ({target_path.stat().st_size:,} bytes)")


for filename, url in ABCD_FILES.items():
    download_if_missing(url, RAW_DIR / filename)

manifest = {
    "source_repo": "https://github.com/asappresearch/abcd",
    "raw_dir": str(RAW_DIR),
    "processed_dir": str(PROCESSED_DIR),
    "eda_dir": str(EDA_DIR),
    "files": {filename: {"url": url, "path": str(RAW_DIR / filename)} for filename, url in ABCD_FILES.items()},
}

(OUTPUT_DIR / "download_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest saved to: {OUTPUT_DIR / 'download_manifest.json'}")

In [ ]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


with gzip.open(RAW_DIR / "abcd_v1.1.json.gz", "rt", encoding="utf-8") as f:
    abcd_data = json.load(f)

guidelines = load_json(RAW_DIR / "guidelines.json")
kb = load_json(RAW_DIR / "kb.json")
ontology = load_json(RAW_DIR / "ontology.json")
utterances = load_json(RAW_DIR / "utterances.json")
sample_data = load_json(RAW_DIR / "abcd_sample.json")

print("ABCD split sizes:")
for split_name, conversations in abcd_data.items():
    print(f"- {split_name}: {len(conversations):,} conversations")

print(f"Guidelines type: {type(guidelines).__name__}")
print(f"KB type: {type(kb).__name__}")
print(f"Ontology type: {type(ontology).__name__}")
print(f"Utterances: {len(utterances):,}")

## Normalize Conversations

The main exported tables use ABCD's `delexed` turns. The original conversations and scenario data include fictional personal fields; this notebook keeps the main EDA tables delexed and avoids exporting names, emails, phone numbers, and account IDs.

In [ ]:
def safe_str(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    return str(value)


def word_count(text: str) -> int:
    text = safe_str(text).strip()
    return len(text.split()) if text else 0


def get_nested(mapping: dict, *keys, default=None):
    value = mapping
    for key in keys:
        if not isinstance(value, dict):
            return default
        value = value.get(key, default)
    return value


def normalize_targets(targets):
    if not isinstance(targets, list):
        targets = []
    padded = (targets + [None, None, None, [], -1])[:5]
    return {
        "target_intent": padded[0],
        "next_step": padded[1],
        "action": padded[2],
        "slot_values": padded[3] if isinstance(padded[3], list) else [],
        "utterance_rank_target": padded[4],
    }


def extract_original_text(original_turns) -> str:
    pieces = []
    for turn in original_turns or []:
        if isinstance(turn, (list, tuple)) and len(turn) >= 2:
            pieces.append(safe_str(turn[1]))
        elif isinstance(turn, dict):
            pieces.append(safe_str(turn.get("text")))
    return " ".join(piece for piece in pieces if piece).strip()


def speaker_counts(delexed_turns) -> dict:
    counts = Counter(safe_str(turn.get("speaker", "unknown")) for turn in delexed_turns or [])
    return {
        "agent_turns": counts.get("agent", 0),
        "customer_turns": counts.get("customer", 0),
        "action_turns": counts.get("action", 0),
        "unknown_turns": counts.get("unknown", 0),
    }


conversation_rows = []
turn_rows = []
action_rows = []

for split_name, conversations in abcd_data.items():
    for convo in conversations:
        convo_id = convo.get("convo_id")
        scenario = convo.get("scenario") or {}
        delexed_turns = convo.get("delexed") or []
        original_turns = convo.get("original") or []

        flow = safe_str(scenario.get("flow"))
        subflow = safe_str(scenario.get("subflow"))
        order = scenario.get("order") or {}
        product = scenario.get("product") or {}
        personal = scenario.get("personal") or {}
        counts = speaker_counts(delexed_turns)

        delexed_text = " ".join(safe_str(turn.get("text")) for turn in delexed_turns if safe_str(turn.get("text"))).strip()
        original_text = extract_original_text(original_turns)
        action_sequence = []
        next_step_sequence = []

        for turn in delexed_turns:
            turn_targets = normalize_targets(turn.get("targets"))
            if turn_targets["action"]:
                action_sequence.append(safe_str(turn_targets["action"]))
            if turn_targets["next_step"]:
                next_step_sequence.append(safe_str(turn_targets["next_step"]))

        convo_uid = f"{split_name}-{convo_id}"
        conversation_rows.append({
            "convo_uid": convo_uid,
            "split": split_name,
            "convo_id": convo_id,
            "flow": flow,
            "subflow": subflow,
            "member_level": safe_str(personal.get("member_level")),
            "payment_method": safe_str(order.get("payment_method")),
            "purchase_date": safe_str(order.get("purchase_date")),
            "num_products": safe_str(order.get("num_products")),
            "product_amounts": json.dumps(product.get("amounts", []), ensure_ascii=False),
            "num_delexed_turns": len(delexed_turns),
            "num_original_turns": len(original_turns),
            "agent_turns": counts["agent_turns"],
            "customer_turns": counts["customer_turns"],
            "action_turns": counts["action_turns"],
            "delexed_word_count": word_count(delexed_text),
            "action_sequence": " > ".join(action_sequence),
            "next_step_sequence": " > ".join(next_step_sequence),
            "delexed_text": delexed_text,
            "original_text_word_count": word_count(original_text),
        })

        for turn in delexed_turns:
            turn_targets = normalize_targets(turn.get("targets"))
            turn_row = {
                "convo_uid": convo_uid,
                "split": split_name,
                "convo_id": convo_id,
                "flow": flow,
                "subflow": subflow,
                "speaker": safe_str(turn.get("speaker")),
                "turn_count": turn.get("turn_count"),
                "text": safe_str(turn.get("text")),
                "word_count": word_count(turn.get("text")),
                "candidate_count": len(turn.get("candidates") or []),
                **turn_targets,
            }
            turn_row["slot_values_json"] = json.dumps(turn_row.pop("slot_values"), ensure_ascii=False)
            turn_rows.append(turn_row)

            if turn_row["speaker"] == "action" or turn_row["action"]:
                action_rows.append(turn_row.copy())

conversation_df = pd.DataFrame(conversation_rows)
turn_df = pd.DataFrame(turn_rows)
action_df = pd.DataFrame(action_rows)

conversation_df.to_parquet(PROCESSED_DIR / "abcd_conversation_summary.parquet", index=False)
turn_df.to_parquet(PROCESSED_DIR / "abcd_turns_delexed.parquet", index=False)
action_df.to_parquet(PROCESSED_DIR / "abcd_action_turns.parquet", index=False)

conversation_df.to_csv(PROCESSED_DIR / "abcd_conversation_summary.csv", index=False)
action_df.to_csv(PROCESSED_DIR / "abcd_action_turns.csv", index=False)

print(f"Conversation rows: {len(conversation_df):,}")
print(f"Turn rows: {len(turn_df):,}")
print(f"Action rows: {len(action_df):,}")
display(conversation_df.head())
display(turn_df.head())

## Dataset Shape EDA

In [ ]:
split_summary = (
    conversation_df.groupby("split", dropna=False)
    .agg(
        conversations=("convo_uid", "count"),
        avg_turns=("num_delexed_turns", "mean"),
        p50_turns=("num_delexed_turns", "median"),
        p95_turns=("num_delexed_turns", lambda s: s.quantile(0.95)),
        avg_action_turns=("action_turns", "mean"),
        avg_customer_turns=("customer_turns", "mean"),
        avg_agent_turns=("agent_turns", "mean"),
    )
    .reset_index()
)

flow_summary = (
    conversation_df.groupby(["flow"], dropna=False)
    .agg(
        conversations=("convo_uid", "count"),
        subflows=("subflow", "nunique"),
        avg_turns=("num_delexed_turns", "mean"),
        avg_action_turns=("action_turns", "mean"),
    )
    .reset_index()
    .sort_values("conversations", ascending=False)
)

subflow_summary = (
    conversation_df.groupby(["flow", "subflow"], dropna=False)
    .agg(
        conversations=("convo_uid", "count"),
        avg_turns=("num_delexed_turns", "mean"),
        avg_action_turns=("action_turns", "mean"),
    )
    .reset_index()
    .sort_values("conversations", ascending=False)
)

speaker_summary = (
    turn_df.groupby(["split", "speaker"], dropna=False)
    .size()
    .reset_index(name="turns")
    .sort_values(["split", "turns"], ascending=[True, False])
)

for name, df in {
    "abcd_split_summary.csv": split_summary,
    "abcd_flow_summary.csv": flow_summary,
    "abcd_subflow_summary.csv": subflow_summary,
    "abcd_speaker_summary.csv": speaker_summary,
}.items():
    df.to_csv(EDA_DIR / name, index=False)

display(split_summary)
display(flow_summary.head(20))
display(subflow_summary.head(30))
display(speaker_summary)

## Action / Policy Sequence EDA

In [ ]:
next_step_summary = (
    turn_df.groupby(["speaker", "next_step"], dropna=False)
    .size()
    .reset_index(name="turns")
    .sort_values("turns", ascending=False)
)

action_summary = (
    action_df.groupby("action", dropna=False)
    .agg(
        turns=("convo_uid", "count"),
        conversations=("convo_uid", "nunique"),
        flows=("flow", "nunique"),
        subflows=("subflow", "nunique"),
    )
    .reset_index()
    .sort_values("turns", ascending=False)
)

action_by_flow = (
    action_df.groupby(["flow", "action"], dropna=False)
    .size()
    .reset_index(name="turns")
    .sort_values(["flow", "turns"], ascending=[True, False])
)

sequence_summary = (
    conversation_df.groupby(["flow", "subflow", "action_sequence"], dropna=False)
    .agg(conversations=("convo_uid", "count"), avg_turns=("num_delexed_turns", "mean"))
    .reset_index()
    .sort_values(["conversations", "flow"], ascending=[False, True])
)

for name, df in {
    "abcd_next_step_summary.csv": next_step_summary,
    "abcd_action_summary.csv": action_summary,
    "abcd_action_by_flow.csv": action_by_flow,
    "abcd_action_sequence_summary.csv": sequence_summary,
}.items():
    df.to_csv(EDA_DIR / name, index=False)

display(next_step_summary.head(30))
display(action_summary.head(30))
display(action_by_flow.head(50))
display(sequence_summary.head(30))

## FinDisputeEval-Oriented EDA

This section marks ABCD conversations by structural role for FinDisputeEval. The goal is to extract reusable dialogue patterns, not financial-dispute labels.

In [ ]:
KEYWORD_GROUPS = {
    "return_refund_policy": [r"\breturn\b", r"\brefund\b", r"exchange", r"wrong size", r"defect", r"replacement"],
    "payment_card_boundary": [r"credit card", r"payment method", r"charged", r"\bcard\b", r"\bpayment\b"],
    "identity_verification": [r"account id", r"username", r"email", r"phone", r"order id", r"membership", r"address"],
    "policy_denial_explanation": [r"unfortunately", r"cannot", r"can't", r"not eligible", r"policy", r"more than", r"within"],
    "escalation_handoff": [r"escalate", r"manager", r"supervisor", r"notify", r"team", r"call you"],
    "order_transaction_context": [r"order", r"purchase", r"shipping", r"delivery", r"package", r"product"],
    "customer_emotion_pressure": [r"angry", r"upset", r"frustrated", r"really", r"please", r"complain", r"sorry"],
}

for group_name, patterns in KEYWORD_GROUPS.items():
    regex = "|".join(patterns)
    conversation_df[f"kw_{group_name}"] = conversation_df["delexed_text"].str.lower().str.contains(regex, regex=True, na=False)

keyword_cols = [f"kw_{name}" for name in KEYWORD_GROUPS]
conversation_df["keyword_hit_count"] = conversation_df[keyword_cols].sum(axis=1)


def assign_findispute_role(row) -> str:
    if row["kw_escalation_handoff"]:
        return "escalation_handoff_template"
    if row["kw_policy_denial_explanation"]:
        return "policy_denial_explanation_template"
    if row["kw_identity_verification"]:
        return "identity_verification_slot_collection"
    if row["kw_return_refund_policy"]:
        return "return_refund_policy_flow"
    if row["kw_payment_card_boundary"]:
        return "payment_card_boundary_negative"
    return "generic_policy_constrained_support"


conversation_df["findispute_structural_role"] = conversation_df.apply(assign_findispute_role, axis=1)

role_summary = (
    conversation_df.groupby(["findispute_structural_role"], dropna=False)
    .agg(conversations=("convo_uid", "count"), avg_turns=("num_delexed_turns", "mean"), avg_action_turns=("action_turns", "mean"))
    .reset_index()
    .sort_values("conversations", ascending=False)
)

keyword_summary = (
    conversation_df.melt(
        id_vars=["split", "flow", "subflow", "findispute_structural_role"],
        value_vars=keyword_cols,
        var_name="keyword_group",
        value_name="hit",
    )
    .query("hit")
    .groupby(["keyword_group", "flow"], dropna=False)
    .size()
    .reset_index(name="conversations")
    .sort_values(["keyword_group", "conversations"], ascending=[True, False])
)

role_by_flow = (
    conversation_df.groupby(["findispute_structural_role", "flow"], dropna=False)
    .size()
    .reset_index(name="conversations")
    .sort_values(["findispute_structural_role", "conversations"], ascending=[True, False])
)

conversation_df.to_parquet(PROCESSED_DIR / "abcd_conversation_summary_with_findispute_roles.parquet", index=False)
conversation_df.to_csv(PROCESSED_DIR / "abcd_conversation_summary_with_findispute_roles.csv", index=False)
role_summary.to_csv(EDA_DIR / "abcd_findispute_role_summary.csv", index=False)
keyword_summary.to_csv(EDA_DIR / "abcd_findispute_keyword_summary.csv", index=False)
role_by_flow.to_csv(EDA_DIR / "abcd_findispute_role_by_flow.csv", index=False)

display(role_summary)
display(keyword_summary.head(40))
display(role_by_flow.head(40))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 11))

split_summary.set_index("split")["conversations"].plot(kind="bar", ax=axes[0, 0], title="ABCD conversations by split")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("conversations")

flow_summary.head(15).sort_values("conversations").plot(
    kind="barh", x="flow", y="conversations", ax=axes[0, 1], legend=False, title="Top flows"
)
axes[0, 1].set_xlabel("conversations")
axes[0, 1].set_ylabel("")

action_summary.head(20).sort_values("turns").plot(
    kind="barh", x="action", y="turns", ax=axes[1, 0], legend=False, title="Top action turns"
)
axes[1, 0].set_xlabel("turns")
axes[1, 0].set_ylabel("")

role_summary.sort_values("conversations").plot(
    kind="barh", x="findispute_structural_role", y="conversations", ax=axes[1, 1], legend=False, title="FinDispute structural role coverage"
)
axes[1, 1].set_xlabel("conversations")
axes[1, 1].set_ylabel("")

plt.tight_layout()
overview_path = EDA_DIR / "abcd_findispute_eda_overview.png"
fig.savefig(overview_path, dpi=160, bbox_inches="tight")
print(f"Saved plot to: {overview_path}")
plt.show()

## FinDispute Seed Samples

These samples are for structural reuse: policy denial language, escalation handoff, identity verification, and action sequence templates. They should not be treated as banking-dispute ground truth.

In [ ]:
sample_specs = [
    ("escalation_handoff_template", 30),
    ("policy_denial_explanation_template", 30),
    ("identity_verification_slot_collection", 30),
    ("return_refund_policy_flow", 30),
    ("payment_card_boundary_negative", 20),
    ("generic_policy_constrained_support", 20),
]

seed_parts = []
used = set()

for role_name, target_n in sample_specs:
    pool = conversation_df[
        conversation_df["findispute_structural_role"].eq(role_name)
        & ~conversation_df["convo_uid"].isin(used)
    ]
    if pool.empty:
        print(f"No rows for role: {role_name}")
        continue
    n = min(target_n, len(pool))
    sampled = pool.sample(n=n, random_state=RANDOM_STATE).copy()
    sampled["seed_bucket"] = role_name
    used.update(sampled["convo_uid"].tolist())
    seed_parts.append(sampled)

if seed_parts:
    seed_conversations = pd.concat(seed_parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    seed_conversations = conversation_df.sample(n=min(100, len(conversation_df)), random_state=RANDOM_STATE).copy()
    seed_conversations["seed_bucket"] = "fallback_random"

seed_convo_cols = [
    "seed_bucket", "convo_uid", "split", "flow", "subflow", "findispute_structural_role",
    "payment_method", "num_delexed_turns", "action_turns", "action_sequence", "delexed_text",
]
seed_turns = turn_df[turn_df["convo_uid"].isin(seed_conversations["convo_uid"])].copy()
seed_turns = seed_turns.merge(seed_conversations[["convo_uid", "seed_bucket", "findispute_structural_role"]], on="convo_uid", how="left")

seed_conversation_path = EDA_DIR / "abcd_findispute_seed_conversations.csv"
seed_turn_path = EDA_DIR / "abcd_findispute_seed_turns.csv"
seed_conversations[seed_convo_cols].to_csv(seed_conversation_path, index=False)
seed_turns.to_csv(seed_turn_path, index=False)

print(f"Saved seed conversations: {seed_conversation_path}")
print(f"Saved seed turns: {seed_turn_path}")
print(f"Seed conversations: {len(seed_conversations):,}")
print(f"Seed turns: {len(seed_turns):,}")
display(seed_conversations[seed_convo_cols].head(20))
display(seed_turns.head(30))

## Expected Output Structure

```text
dataset/external/abcd/raw/
  abcd_v1.1.json.gz
  abcd_sample.json
  guidelines.json
  kb.json
  ontology.json
  utterances.json

dataset/interim/abcd/processed_findispute/
  abcd_conversation_summary.csv / .parquet
  abcd_turns_delexed.parquet
  abcd_action_turns.csv / .parquet
  abcd_conversation_summary_with_findispute_roles.csv / .parquet

outputs/data_pipeline/abcd_structure_eda/eda_v01/<run_id>/
  abcd_split_summary.csv
  abcd_flow_summary.csv
  abcd_subflow_summary.csv
  abcd_action_summary.csv
  abcd_action_sequence_summary.csv
  abcd_findispute_role_summary.csv
  abcd_findispute_keyword_summary.csv
  abcd_findispute_eda_overview.png
  abcd_findispute_seed_conversations.csv
  abcd_findispute_seed_turns.csv
```
